# PREPROCESS FOR LAW READING COMPREHENSION

Create grounded question-context pairs from the Law Reading Comprehension dataset and insert them into PostgreSQL.

In [1]:
import sys
from pathlib import Path

from datasets import load_from_disk

project_dir = Path.cwd()
if not (project_dir / "data").exists():
    project_dir = project_dir.parent
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

data_dir = project_dir / "data" / "Legal" / "law_reading_comphresion"
dataset_dict = load_from_disk(str(data_dir))
dataset = dataset_dict["train"]

print(dataset)
print(dataset.features)

/home/nhminh/AI_Project/VietRAG-Embed-E5-Base/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['text', 'meta', 'content', 'citation', 'qas', 'task_type'],
    num_rows: 894619
})
{'text': Value(dtype='string', id=None), 'meta': {'issuing_agency': Value(dtype='string', id=None), 'promulgation_date': Value(dtype='string', id=None), 'sign_number': Value(dtype='string', id=None), 'signer': Value(dtype='string', id=None), 'type': Value(dtype='string', id=None)}, 'content': Value(dtype='string', id=None), 'citation': Value(dtype='string', id=None), 'qas': [{'question': Value(dtype='string', id=None), 'answer': Value(dtype='string', id=None)}], 'task_type': Value(dtype='string', id=None)}


In [2]:
import pyarrow.compute as pc

table = dataset.data.table
qa_counts = pc.list_value_length(table["qas"])
content_lengths = pc.utf8_length(pc.utf8_trim_whitespace(table["content"]))
total_qas = pc.sum(qa_counts).as_py()
empty_documents = pc.sum(pc.equal(content_lengths, 0)).as_py()

print(f"Documents: {len(dataset):,}")
print(f"Question-answer pairs: {total_qas:,}")
print(f"Empty documents to skip: {empty_documents:,}")

Documents: 894,619
Question-answer pairs: 1,685,152
Empty documents to skip: 5


In [3]:
import re

CHUNK_SIZE = 384
CHUNK_OVERLAP = 48
MIN_CHUNK_SIZE = CHUNK_SIZE // 2


def chunk_content(content: str) -> list[tuple[int, str]]:
    """Split legal text into overlapping chunks near paragraph or sentence boundaries."""
    content = content.strip()
    if not content:
        return []

    chunks = []
    start = 0

    while start < len(content):
        target_end = min(start + CHUNK_SIZE, len(content))
        end = target_end

        if target_end < len(content):
            window = content[start:target_end]
            boundaries = [
                window.rfind("\n"),
                window.rfind(". "),
                window.rfind("; "),
            ]
            boundary = max(boundaries)
            if boundary >= MIN_CHUNK_SIZE:
                end = start + boundary + 1

        chunk = content[start:end].strip()
        if chunk:
            chunks.append((start, chunk))

        if end >= len(content):
            break
        start = max(end - CHUNK_OVERLAP, start + 1)

    return chunks


def tokenize(text: str) -> set[str]:
    return set(re.findall(r"\w+", text.lower(), flags=re.UNICODE))


def select_positive_chunk(
    question: str,
    answer: str,
    citation: str,
    chunks: list[tuple[int, str]],
) -> tuple[int, str]:
    """Select the source chunk covering the most terms from the labelled QA."""
    qa_tokens = tokenize(f"{question} {answer}")

    def score(chunk: tuple[int, str]) -> tuple[float, int, int]:
        start, text = chunk
        chunk_tokens = tokenize(f"{citation} {text}")
        overlap = len(qa_tokens.intersection(chunk_tokens))
        coverage = overlap / max(len(qa_tokens), 1)
        return coverage, overlap, -start

    return max(chunks, key=score)

In [4]:
import hashlib

ID_PREFIX = "legal_002"
SOURCE = "Law Reading Comprehension QA"


def iter_records(dataset):
    """Yield records without loading all generated pairs into memory."""
    for row in dataset:
        citation = row["citation"].strip()
        content = row["content"].strip()
        chunks = chunk_content(content)

        if not chunks:
            continue

        for qa in row["qas"]:
            question = qa["question"].strip()
            answer = qa["answer"].strip()
            if not question:
                continue

            _, chunk_text = select_positive_chunk(
                question=question,
                answer=answer,
                citation=citation,
                chunks=chunks,
            )
            positive = f"{citation}\n{chunk_text}"
            pair_key = f"{question}|{positive}"
            pair_hash = hashlib.sha1(pair_key.encode("utf-8")).hexdigest()[:24]

            yield {
                "data_id": f"{ID_PREFIX}_{pair_hash}",
                "source": SOURCE,
                "title": citation,
                "anchor": question,
                "positive": positive,
                "hard_negative": None,
            }

In [5]:
from itertools import islice

preview_records = list(islice(iter_records(dataset), 3))
for record in preview_records:
    print(record)

{'data_id': 'legal_002_3f5f14c32deb578ec8b63385', 'source': 'Law Reading Comprehension QA', 'title': 'Điều 67 Luật Người lao động Việt Nam đi làm việc ở nước ngoài theo hợp đồng 2020 số 69/2020/QH14 mới nhất', 'anchor': 'Tóm tắt nội dung của Điều 67 Luật Người lao động Việt Nam đi làm việc ở nước ngoài theo hợp đồng 2020 số 69/2020/QH14 mới nhất', 'positive': 'Điều 67 Luật Người lao động Việt Nam đi làm việc ở nước ngoài theo hợp đồng 2020 số 69/2020/QH14 mới nhất\niệt Nam đi làm việc ở nước ngoài theo hợp đồng;\nđ) Hỗ trợ thân nhân người lao động trong trường hợp người lao động chết, bị mất tích trong thời gian làm việc ở nước ngoài.\n2. Hỗ trợ đối với doanh nghiệp trong trường hợp sau đây:\na) Khai thác, phát triển, ổn định thị trường lao động ngoài nước;\nb) Giải quyết những rủi ro liên quan đến người lao động do mình đưa đi.\n3.', 'hard_negative': None}
{'data_id': 'legal_002_8f220b8ef51a3b92f3535cab', 'source': 'Law Reading Comprehension QA', 'title': 'Điều 1 Luật Hôn nhân và gia 

In [6]:
from sqlalchemy import func, select
from sqlalchemy.dialects.postgresql import insert as postgresql_insert

from database.models import LegalModel
from database.sql_manager import SQL_Manager

INSERT_BATCH_SIZE = 1_000
sql_mng = SQL_Manager()
sql_mng.create_legal_model()
existing_before = sql_mng.con.scalar(
    select(func.count())
    .select_from(LegalModel)
    .where(LegalModel.data_id.like(f"{ID_PREFIX}_%"))
)
print(f"Existing {ID_PREFIX} records: {existing_before:,}")

processed = 0
inserted = 0
batch = []


def insert_batch(records: list[dict]) -> int:
    statement = postgresql_insert(LegalModel).values(records)
    statement = statement.on_conflict_do_nothing(
        index_elements=["data_id"]
    ).returning(LegalModel.data_id)
    inserted_ids = sql_mng.con.scalars(statement).all()
    return len(inserted_ids)


try:
    for record in iter_records(dataset):
        batch.append(record)

        if len(batch) < INSERT_BATCH_SIZE:
            continue

        inserted += insert_batch(batch)
        processed += len(batch)
        sql_mng.con.commit()
        batch.clear()

        if processed % 50_000 == 0:
            print(f"Processed: {processed:,}; inserted: {inserted:,}")

    if batch:
        inserted += insert_batch(batch)
        processed += len(batch)
        sql_mng.con.commit()
except Exception:
    sql_mng.con.rollback()
    raise
finally:
    sql_mng.close()

print(f"Finished. Processed: {processed:,}; inserted: {inserted:,}")

Existing legal_002 records: 0
Processed: 50,000; inserted: 49,958
Processed: 100,000; inserted: 99,936
Processed: 150,000; inserted: 149,915
Processed: 200,000; inserted: 199,897
Processed: 250,000; inserted: 249,844
Processed: 300,000; inserted: 299,742
Processed: 350,000; inserted: 349,206
Processed: 400,000; inserted: 399,198
Processed: 450,000; inserted: 448,673
Processed: 500,000; inserted: 498,652
Processed: 550,000; inserted: 548,055
Processed: 600,000; inserted: 597,977
Processed: 650,000; inserted: 647,957
Processed: 700,000; inserted: 697,351
Processed: 750,000; inserted: 746,803
Processed: 800,000; inserted: 796,073
Processed: 850,000; inserted: 845,639
Processed: 900,000; inserted: 895,335
Processed: 950,000; inserted: 944,477
Processed: 1,000,000; inserted: 994,457
Processed: 1,050,000; inserted: 1,044,444
Processed: 1,100,000; inserted: 1,094,423
Processed: 1,150,000; inserted: 1,144,347
Processed: 1,200,000; inserted: 1,193,696
Processed: 1,250,000; inserted: 1,242,864
P